In [9]:
import numpy as np
import os, gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from scipy import stats as scipy_stats
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
 

In [10]:
# ─── Paths & Hyper-params ─────────────────────────────────────────────────────
DATA_ROOT = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2"
TEST_IN   = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in"
OUTPUT    = "/kaggle/working/preds.npy"
 
BATCH_SIZE  = 2       # T4 safe with AMP
EPOCHS      = 1      # AMP lets us afford a few more
LR          = 3e-4
HIDDEN_DIM  = 96      # unchanged – fits in T4
KERNEL_SIZE = 3
NUM_LAYERS  = 3
HISTORY_PM  = 10
FORECAST    = 16
H, W        = 140, 124
PATIENCE    = 6
 
# Loss weights – tune these; start balanced
ALPHA_EP   = 3.0   # episode SMAPE weight
BETA_CORR  = 1.5   # correlation loss weight
HZ_GAMMA   = 0.15  # horizon difficulty ramp: step t gets weight (1+γ)^t
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print(f"Device: {DEVICE}")
print(f"Phase 2 FADE-ConvLSTM | AMP mixed-precision enabled")
 

Device: cuda
Phase 2 FADE-ConvLSTM | AMP mixed-precision enabled


In [11]:
# ─── Data loading / normalisation (unchanged from base) ───────────────────────
MET_VARS      = ['q2','t2','u10','v10','swdown','pblh','psfc','rain']
EMISSION_VARS = ['PM25','NH3','SO2','NOx','NMVOC_e','NMVOC_finn','bio']
MONTHS        = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
LOG_FEATURES  = {'PM25','NH3','SO2','NOx','NMVOC_e','NMVOC_finn','bio','cpm25'}
 
def load_month(month, base=DATA_ROOT):
    path = os.path.join(base, "raw", month)
    data = {}
    data['cpm25'] = np.load(os.path.join(path, 'cpm25.npy')).astype(np.float32)
    for v in MET_VARS + EMISSION_VARS:
        fpath = os.path.join(path, f'{v}.npy')
        if os.path.exists(fpath):
            data[v] = np.load(fpath).astype(np.float32)
    return data
 
all_data = {}
for m in MONTHS:
    print(f"Loading {m}...")
    all_data[m] = load_month(m)
 
def compute_stats(all_data, key):
    arrays   = [d[key] for d in all_data.values() if key in d]
    combined = np.concatenate(arrays, axis=0).astype(np.float32)
    if key in LOG_FEATURES:
        combined = np.log1p(combined * 1e9)
    return combined.mean(), combined.std() + 1e-8
 
print("Computing normalisation stats...")
norm_stats = {}
for key in ['cpm25'] + MET_VARS + EMISSION_VARS:
    try:
        norm_stats[key] = compute_stats(all_data, key)
    except Exception:
        pass
 
def normalize(arr, key):
    mean, std = norm_stats[key]
    if key in LOG_FEATURES:
        arr = np.log1p(arr.astype(np.float32) * 1e9)
    return (arr - mean) / std
 
def denormalize_pm25(arr_norm):
    mean, std = norm_stats['cpm25']
    return np.expm1(arr_norm * std + mean) / 1e9

Loading APRIL_16...
Loading JULY_16...
Loading OCT_16...
Loading DEC_16...
Computing normalisation stats...


In [12]:
# ─── Episode Identification (unchanged from base) ─────────────────────────────
def identify_episodes(pm_arr, seasonal_period=24):
    from scipy.ndimage import uniform_filter1d
    T, H, W = pm_arr.shape
    pm_2d    = pm_arr.reshape(T, -1).astype(np.float64)
    trend    = uniform_filter1d(pm_2d, size=seasonal_period, axis=0)
    detrended = pm_2d - trend
    seasonal  = np.zeros_like(detrended)
    for h in range(seasonal_period):
        idx = np.arange(h, T, seasonal_period)
        seasonal[idx] = detrended[idx].mean(axis=0, keepdims=True)
    residual = detrended - seasonal
    sigma    = residual.std(axis=0, keepdims=True) + 1e-8
    return (residual > 3 * sigma).reshape(T, H, W)
 
print("Computing episode masks...")
episode_masks = {}
for m, d in all_data.items():
    episode_masks[m] = identify_episodes(d['cpm25'])
    frac = episode_masks[m].mean()
    print(f"  {m}: episodic fraction = {frac*100:.2f}%")
 
 
# ─── Dataset (now also returns future episode mask for loss) ──────────────────
class PM25DatasetPhase2(Dataset):
    def __init__(self, all_data, episode_masks, stride=1):
        self.samples = []   # (pm_hist, met_hist, pm_fut, fut_ep_mask)
        self.weights = []
        other_keys = [k for k in MET_VARS + EMISSION_VARS if k in norm_stats]
 
        for month, data in all_data.items():
            T       = data['cpm25'].shape[0]
            pm      = normalize(data['cpm25'], 'cpm25')
            mets    = np.stack([normalize(data[k], k)
                                for k in other_keys if k in data], axis=1)
            ep_mask = episode_masks[month]
            window  = HISTORY_PM + FORECAST
 
            for t in range(0, T - window + 1, stride):
                pm_hist  = pm[t : t + HISTORY_PM]
                pm_fut   = pm[t + HISTORY_PM : t + window]
                met_hist = mets[t : t + HISTORY_PM]
                fut_ep   = ep_mask[t + HISTORY_PM : t + window].astype(np.float32)
                ep_frac  = fut_ep.mean()
                weight   = 1.0 + 9.0 * ep_frac
 
                self.samples.append((pm_hist, met_hist, pm_fut, fut_ep))
                self.weights.append(weight)
 
        self.weights = np.array(self.weights, dtype=np.float32)
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, idx):
        pm_hist, met_hist, pm_fut, fut_ep = self.samples[idx]
        return (torch.tensor(pm_hist),
                torch.tensor(met_hist),
                torch.tensor(pm_fut),
                torch.tensor(fut_ep))
 
 
# Train / Val split
train_data, val_data, train_ep, val_ep = {}, {}, {}, {}
for m, d in all_data.items():
    T = d['cpm25'].shape[0]; split = int(T * 0.8)
    train_data[m] = {k: v[:split] for k, v in d.items()
                     if isinstance(v, np.ndarray) and v.ndim >= 1 and v.shape[0] == T}
    val_data[m]   = {k: v[split:] for k, v in d.items()
                     if isinstance(v, np.ndarray) and v.ndim >= 1 and v.shape[0] == T}
    train_ep[m]   = episode_masks[m][:split]
    val_ep[m]     = episode_masks[m][split:]
 
train_ds = PM25DatasetPhase2(train_data, train_ep, stride=1)
val_ds   = PM25DatasetPhase2(val_data,   val_ep,   stride=1)
sampler  = WeightedRandomSampler(
    torch.tensor(train_ds.weights), len(train_ds), replacement=True)
 
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                      num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

Computing episode masks...
  APRIL_16: episodic fraction = 0.99%
  JULY_16: episodic fraction = 0.93%
  OCT_16: episodic fraction = 0.85%
  DEC_16: episodic fraction = 1.02%
Train: 2245, Val: 487


In [13]:
# ==============================================================================
#  ARCHITECTURE
# ==============================================================================
 
# ─── 1. FADELayer – Lightweight Fractional Advection-Diffusion Physics ────────
class FADELayer(nn.Module):
    """
    Approximates the Fractional Advection-Diffusion Equation without PDE solvers.
 
    For each input time-step t, computes:
        fade_t = pm_t  +  D * Lap(pm_t)  +  alpha * (pm_t - frac_mem_t)
 
    Where:
        - D * Lap(pm)   ≈ learnable diffusion via Laplacian convolution
        - frac_mem_t    = weighted average of past states (fractional memory)
        - alpha         = memory coupling coefficient (learnable)
 
    The output is a PHYSICS-INFORMED version of the raw PM2.5 sequence,
    passed to the spatial encoder — zero PDE solve required.
 
    Why this helps:
        - Diffusion term smooths local noise, improving Global SMAPE
        - Fractional memory term captures long-range temporal correlations
          that ConvLSTM alone misses, improving Episode Correlation
        - Learnable D and alpha let the model decide how much physics to trust
    """
    def __init__(self, memory_steps: int = 3):
        super().__init__()
        self.memory_steps = memory_steps  # how many past steps to average (budget: 2-3)
 
        # Learnable diffusion coefficient (initialised small, positive via softplus)
        self._log_D    = nn.Parameter(torch.tensor(-2.0))   # D ≈ 0.135 initially
        # Learnable memory coupling (initialised small)
        self._log_alpha = nn.Parameter(torch.tensor(-2.5))  # alpha ≈ 0.082 initially
 
        # Fixed Laplacian kernel (isotropic finite-difference)
        lap_kernel = torch.tensor([[[[0,  1, 0],
                                     [1, -4, 1],
                                     [0,  1, 0]]]], dtype=torch.float32)
        self.register_buffer('lap_kernel', lap_kernel)
 
        # Learnable memory weights (softmax-normalised so they sum to 1)
        self.mem_logits = nn.Parameter(torch.zeros(memory_steps))
 
    @property
    def D(self):
        return F.softplus(self._log_D)
 
    @property
    def alpha(self):
        return F.softplus(self._log_alpha)
 
    def laplacian(self, pm: torch.Tensor) -> torch.Tensor:
        """pm: (B, 1, H, W) → (B, 1, H, W)"""
        return F.conv2d(pm, self.lap_kernel, padding=1)
 
    def forward(self, pm_seq: torch.Tensor) -> torch.Tensor:
        """
        pm_seq: (B, T, H, W)  raw normalised PM2.5 history
        returns: (B, T, H, W) physics-informed sequence
        """
        B, T, H, W = pm_seq.shape
        out = []
        mem_weights = F.softmax(self.mem_logits, dim=0)  # (memory_steps,)
 
        for t in range(T):
            pm_t = pm_seq[:, t:t+1]                      # (B, 1, H, W)
 
            # Diffusion term
            diff_term = self.D * self.laplacian(pm_t)    # (B, 1, H, W)
 
            # Fractional memory term: weighted mean of available past steps
            n_past = min(t, self.memory_steps)
            if n_past > 0:
                w = F.softmax(self.mem_logits[:n_past], dim=0)
                frac_mem = sum(
                    w[j] * pm_seq[:, t - n_past + j : t - n_past + j + 1]
                    for j in range(n_past)
                )
                mem_term = self.alpha * (pm_t - frac_mem)
            else:
                mem_term = torch.zeros_like(pm_t)
 
            # FADE-enhanced step
            fade_t = pm_t + diff_term + mem_term
            out.append(fade_t)
 
        return torch.cat(out, dim=1)                      # (B, T, H, W)
 
 
# ─── 2. ConvLSTM Cell (unchanged from base) ───────────────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel_size):
        super().__init__()
        self.hidden_ch = hidden_ch
        pad = kernel_size // 2
        self.conv = nn.Conv2d(in_ch + hidden_ch, 4 * hidden_ch, kernel_size, padding=pad)
        self.ln   = nn.GroupNorm(1, 4 * hidden_ch)
 
    def forward(self, x, h, c):
        gates = self.ln(self.conv(torch.cat([x, h], dim=1)))
        i, f, g, o = gates.chunk(4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c
 
    def init_hidden(self, B, H, W, device):
        return (torch.zeros(B, self.hidden_ch, H, W, device=device),
                torch.zeros(B, self.hidden_ch, H, W, device=device))
 
 
# ─── 3. Multi-scale Spatial Encoder (unchanged from base) ─────────────────────
class SpatialEncoder(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1, dilation=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=2, dilation=2), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=4, dilation=4), nn.ReLU())
        self.fuse = nn.Conv2d(out_ch * 3, out_ch, 1)
 
    def forward(self, x):
        return self.fuse(torch.cat([self.enc1(x), self.enc2(x), self.enc3(x)], dim=1))
 
 
# ─── 4. Spatial Attention (unchanged from base) ───────────────────────────────
class SpatialAttention(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, in_ch // 4, 1), nn.ReLU(),
            nn.Conv2d(in_ch // 4, 1, 1),     nn.Sigmoid())
    def forward(self, x):
        return x * self.conv(x)
 
 
# ─── 5. Wind Warp (unchanged from base) ───────────────────────────────────────
class WindWarp(nn.Module):
    def __init__(self, u_idx, v_idx, scale=0.05):
        super().__init__()
        self.u_idx = u_idx; self.v_idx = v_idx; self.scale = scale
 
    def forward(self, pm, last_met):
        if self.u_idx is None or self.v_idx is None:
            return pm
        B, _, H, W = pm.shape
        u = last_met[:, self.u_idx:self.u_idx+1]
        v = last_met[:, self.v_idx:self.v_idx+1]
        xs = torch.linspace(-1, 1, W, device=pm.device)
        ys = torch.linspace(-1, 1, H, device=pm.device)
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
        grid  = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0).expand(B, -1, -1, -1)
        flow  = torch.stack([u.squeeze(1), v.squeeze(1)], dim=-1) * self.scale
        return F.grid_sample(pm, (grid + flow).clamp(-1, 1),
                             align_corners=False, mode='bilinear', padding_mode='border')
 
 
# ─── 6. EpisodeDetectorV2 – multi-scale spike localiser ──────────────────────
class EpisodeDetectorV2(nn.Module):
    """
    Improved over base EpisodeDetector:
    - Multi-scale context (two dilation rates) captures both local spikes
      and broader pollution plumes
    - Sigmoid output = soft episode probability map used by decoder
    Why it helps Episode Correlation: forces the model to explicitly attend
    to where spikes will occur, boosting shape/timing accuracy.
    """
    def __init__(self, in_ch: int):
        super().__init__()
        mid = max(in_ch // 4, 16)
        self.local  = nn.Sequential(
            nn.Conv2d(in_ch, mid, 3, padding=1, dilation=1), nn.ReLU())
        self.nonlocal_ = nn.Sequential(
            nn.Conv2d(in_ch, mid, 3, padding=3, dilation=3), nn.ReLU())
        self.fuse = nn.Sequential(
            nn.Conv2d(mid * 2, mid, 1), nn.ReLU(),
            nn.Conv2d(mid, 1, 1),       nn.Sigmoid())
 
    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.fuse(torch.cat([self.local(h), self.nonlocal_(h)], dim=1))
 
 
# ─── 7. FADE-ConvLSTM Phase 2 Model ──────────────────────────────────────────
class FADEConvLSTM(nn.Module):
    """
    Architecture: FADELayer → SpatialEncoder+ConvLSTM Encoder → Decoder
 
    FADE layer is applied to PM2.5 history BEFORE the encoder, injecting
    physics priors (diffusion, fractional memory) into every encoder step.
    The rest of the architecture keeps the proven base structure while adding:
      - EpisodeDetectorV2 for better spike localisation
      - Episode-conditioned amplitude boosting in the decoder
      - Gradient checkpointing on the encoder (saves VRAM)
    """
    def __init__(self, met_channels, hidden_dim, kernel_size, num_layers,
                 u_idx, v_idx, use_checkpoint=True):
        super().__init__()
        self.num_layers     = num_layers
        self.hidden_dim     = hidden_dim
        self.use_checkpoint = use_checkpoint
        enc_met_ch          = hidden_dim // 2
 
        # Physics prior (FADE)
        self.fade         = FADELayer(memory_steps=3)
        self.spatial_enc  = SpatialEncoder(met_channels, enc_met_ch)
        self.wind_warp    = WindWarp(u_idx, v_idx, scale=0.05)
        self.episode_det  = EpisodeDetectorV2(hidden_dim)
 
        # Encoder / Decoder ConvLSTM stacks
        self.enc_cells = nn.ModuleList()
        self.dec_cells = nn.ModuleList()
        for i in range(num_layers):
            enc_in = (1 + enc_met_ch) if i == 0 else hidden_dim
            dec_in = (1 + enc_met_ch + 1) if i == 0 else hidden_dim   # +1 for ep_map
            self.enc_cells.append(ConvLSTMCell(enc_in, hidden_dim, kernel_size))
            self.dec_cells.append(ConvLSTMCell(dec_in, hidden_dim, kernel_size))
 
        self.attn = SpatialAttention(hidden_dim)
 
        # Output head (base residual)
        self.output_head = nn.Sequential(
            nn.Conv2d(hidden_dim, 64, 3, padding=1), nn.ReLU(),
            nn.Dropout2d(0.1),
            nn.Conv2d(64, 32, 3, padding=1),         nn.ReLU(),
            nn.Conv2d(32, 1, 1))
 
        # Episode amplitude booster (Softplus keeps output ≥ 0)
        self.episode_amp = nn.Sequential(
            nn.Conv2d(hidden_dim, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 1, 1),                     nn.Softplus())
 
    def _encode_step(self, x, states, i, cell):
        """Wrapper so torch.utils.checkpoint can trace it."""
        h, c = states[i]
        h, c = cell(x, h, c)
        return h, c
 
    def forward(self, pm_hist, met_hist):
        """
        pm_hist  : (B, 10, H, W)
        met_hist : (B, 10, C, H, W)
        returns  : (B, 16, H, W)
        """
        B, T_hist, H, W = pm_hist.shape
 
        # ── FADE physics prior ──────────────────────────────
        fade_pm = self.fade(pm_hist)   # (B, 10, H, W)  physics-enhanced
 
        # ── Encoder ────────────────────────────────────────
        states = [cell.init_hidden(B, H, W, pm_hist.device)
                  for cell in self.enc_cells]
 
        for t in range(T_hist):
            pm_t    = fade_pm[:, t:t+1]            # FADE-enhanced PM
            enc_met = self.spatial_enc(met_hist[:, t])
            x       = torch.cat([pm_t, enc_met], dim=1)
 
            for i, cell in enumerate(self.enc_cells):
                if self.use_checkpoint and self.training:
                    # Gradient checkpointing – recompute activations instead
                    # of storing them, trading compute for VRAM (~25% saving)
                    import torch.utils.checkpoint as cp
                    h, c = cp.checkpoint(cell, x, states[i][0], states[i][1],
                                         use_reentrant=False)
                else:
                    h, c = cell(x, states[i][0], states[i][1])
                states[i] = (h, c)
                x = h
 
        # ── Decoder ────────────────────────────────────────
        last_met   = met_hist[:, -1]                # (B, C, H, W) – t=9
        dec_states = states
        preds      = []
        prev_pm    = pm_hist[:, -1:].clone()        # raw (not FADE) for decoding
 
        for t in range(FORECAST):
            warped  = self.wind_warp(prev_pm, last_met)
            enc_met = self.spatial_enc(last_met)     # constant across decoder steps
            h_cur   = dec_states[-1][0]
            ep_map  = self.episode_det(h_cur)        # (B, 1, H, W)
 
            x = torch.cat([warped, enc_met, ep_map], dim=1)
            for i, cell in enumerate(self.dec_cells):
                h, c = cell(x, dec_states[i][0], dec_states[i][1])
                dec_states[i] = (h, c)
                x = h
 
            x      = self.attn(x)
            delta  = self.output_head(x)
            amp    = self.episode_amp(x)
            pred_t = prev_pm + delta + ep_map * amp
            preds.append(pred_t)
            prev_pm = pred_t
 
        return torch.cat(preds, dim=1)   # (B, 16, H, W)

In [14]:
# ==============================================================================
#  LOSS FUNCTION
# ==============================================================================
 
class EpisodeAwareLoss(nn.Module):
    """
    Composite loss aligned with competition metrics:
 
        L = SMAPE_global  +  α * SMAPE_episode  +  β * (1 - Pearson_episode)
              ↓                   ↓                        ↓
          Global SMAPE        Episode SMAPE           Episode Correlation
          metric surrogate    metric surrogate          metric surrogate
 
    Also applies multi-horizon weighting: later timesteps (harder) get
    exponentially larger weight  w_t = (1 + γ)^t so the model is not lazy
    about the last few steps of the 16-step horizon.
 
    Implementation notes:
    - SMAPE is computed as 2|y-ŷ| / (|y|+|ŷ|+ε), numerically safe
    - Pearson correlation loss = 1 - r  (in [0, 2], lower is better)
    - Episode mask is soft (float) so gradients flow through it cleanly
    """
 
    def __init__(self, alpha: float = ALPHA_EP,
                 beta:  float = BETA_CORR,
                 hz_gamma: float = HZ_GAMMA,
                 eps: float = 1e-6):
        super().__init__()
        self.alpha    = alpha
        self.beta     = beta
        self.eps      = eps
        # Multi-horizon weights: (1 + γ)^0 … (1 + γ)^15, normalised
        w = torch.tensor([(1 + hz_gamma) ** t for t in range(FORECAST)],
                         dtype=torch.float32)
        self.register_buffer('hz_weights', w / w.sum() * FORECAST)
 
    @staticmethod
    def smape(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
        """Element-wise SMAPE ∈ [0, 2]."""
        return 2 * (pred - target).abs() / (pred.abs() + target.abs() + eps)
 
    @staticmethod
    def pearson_loss(pred: torch.Tensor, target: torch.Tensor,
                     eps: float = 1e-6) -> torch.Tensor:
        """
        1 - Pearson r over flattened (pred, target) pair.
        If there are fewer than 2 valid points, returns 0.
        """
        pred_f   = pred.reshape(-1)
        target_f = target.reshape(-1)
        if pred_f.numel() < 2:
            return pred_f.new_tensor(0.0)
        vp = pred_f   - pred_f.mean()
        vt = target_f - target_f.mean()
        r  = (vp * vt).sum() / (vp.norm() * vt.norm() + eps)
        return 1.0 - r
 
    def forward(self, pred, target, ep_mask):
        hz = self.hz_weights.to(pred.device).view(1, FORECAST, 1, 1)
 
        # ── Global SMAPE ────────────────────────────────────
        smape_all = self.smape(pred, target, self.eps)           # (B,16,H,W)
        loss_global = (smape_all * hz).mean()
 
        # ── Episode-masked SMAPE ────────────────────────────
        # ep_mask is float, so this is a soft weighted mean
        ep_weight   = ep_mask * hz                               # (B,16,H,W)
        ep_denom    = ep_weight.sum() + self.eps
        loss_episode = (smape_all * ep_weight).sum() / ep_denom
 
        # ── Episode Pearson Correlation loss ────────────────
        # Flatten over all episode pixels across batch
        ep_bool     = (ep_mask > 0.5)                           # hard mask for corr
        pred_ep     = pred[ep_bool]
        target_ep   = target[ep_bool]
        loss_corr   = self.pearson_loss(pred_ep, target_ep, self.eps)
 
        return loss_global + self.alpha * loss_episode + self.beta * loss_corr

In [15]:
# ==============================================================================
#  BUILD MODEL
# ==============================================================================
 
sample_pm, sample_met, _, _ = train_ds[0]
C_met      = sample_met.shape[1]
other_keys = [k for k in MET_VARS + EMISSION_VARS if k in norm_stats]
u_idx      = other_keys.index('u10') if 'u10' in other_keys else None
v_idx      = other_keys.index('v10') if 'v10' in other_keys else None
print(f"Met channels: {C_met}, u_idx: {u_idx}, v_idx: {v_idx}")
 
_model = FADEConvLSTM(C_met, HIDDEN_DIM, KERNEL_SIZE, NUM_LAYERS,
                      u_idx, v_idx, use_checkpoint=False)  # see note below
_model = _model.to(DEVICE)

num_gpus = torch.cuda.device_count()
if num_gpus > 1:
    model = nn.DataParallel(_model, device_ids=list(range(num_gpus)))
    print(f"DataParallel on {num_gpus} GPUs")
else:
    model = _model
    print("Single GPU")
total_params = sum(p.numel() for p in _model.parameters() if p.requires_grad)
print(f"FADE-ConvLSTM parameters: {total_params:,}")
 
# Print FADE physics params
print(f"Initial FADE D (diffusion):  {_model.fade.D.item():.4f}")
print(f"Initial FADE α (mem coupling): {_model.fade.alpha.item():.4f}")

Met channels: 15, u_idx: 2, v_idx: 3
DataParallel on 2 GPUs
FADE-ConvLSTM parameters: 3,840,009
Initial FADE D (diffusion):  0.1269
Initial FADE α (mem coupling): 0.0789


In [16]:

 
criterion = EpisodeAwareLoss(alpha=ALPHA_EP, beta=BETA_CORR, hz_gamma=HZ_GAMMA).to(DEVICE)
optimizer = torch.optim.AdamW(_model.parameters(), lr=LR, weight_decay=1e-4)
 
# Warmup (3 epochs) → Cosine decay
sched_warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=3)
sched_cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS - 3, eta_min=LR * 0.05)
scheduler    = SequentialLR(optimizer, schedulers=[sched_warmup, sched_cosine],
                             milestones=[3])
 
scaler     = GradScaler()   # AMP gradient scaler
best_val   = float('inf')
patience_c = 0
 
for epoch in range(1, EPOCHS + 1):
    # ── Train ───────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for pm_hist, met_hist, pm_fut, fut_ep in tqdm(train_dl, desc=f"Ep {epoch} train"):
        pm_hist = pm_hist.to(DEVICE)
        met_hist = met_hist.to(DEVICE)
        pm_fut   = pm_fut.to(DEVICE)
        fut_ep   = fut_ep.to(DEVICE)
 
        optimizer.zero_grad(set_to_none=True)
 
        with autocast():                     # mixed precision forward pass
            pred = model(pm_hist, met_hist)  # (B, 16, H, W)
            loss = criterion(pred, pm_fut, fut_ep)
 
        scaler.scale(loss).backward()
        # Gradient clipping prevents NaN with AMP
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
 
    scheduler.step()
 
    # ── Validate ────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for pm_hist, met_hist, pm_fut, fut_ep in val_dl:
            pm_hist  = pm_hist.to(DEVICE)
            met_hist = met_hist.to(DEVICE)
            pm_fut   = pm_fut.to(DEVICE)
            fut_ep   = fut_ep.to(DEVICE)
            with autocast():
                pred = model(pm_hist, met_hist)
                loss = criterion(pred, pm_fut, fut_ep)
            val_loss += loss.item()
 
    tl = train_loss / len(train_dl)
    vl = val_loss   / len(val_dl)
    d  = _model.fade.D.item()
    a  = _model.fade.alpha.item()
    print(f"Epoch {epoch:02d}  train={tl:.4f}  val={vl:.4f}  "
          f"FADE D={d:.4f}  α={a:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")
 
    if vl < best_val:
        best_val   = vl
        patience_c = 0
        torch.save(_model.state_dict(), '/kaggle/working/best_fade_model.pt')
        print(f"  ✓ Saved new best model (val={best_val:.4f})")
    else:
        patience_c += 1
        if patience_c >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break
 
    torch.cuda.empty_cache(); gc.collect()

Ep 1 train: 100%|██████████| 1123/1123 [39:03<00:00,  2.09s/it]


Epoch 01  train=2.8685  val=2.3611  FADE D=0.1249  α=0.0804  lr=1.20e-04
  ✓ Saved new best model (val=2.3611)


In [17]:
# ==============================================================================
#  INFERENCE
# ==============================================================================
 
gc.collect(); torch.cuda.empty_cache()
 
_model.load_state_dict(
    torch.load('/kaggle/working/best_fade_model.pt', map_location=DEVICE))
_model = _model.to(DEVICE)
_model.eval()
_model.use_checkpoint = False   # no checkpointing at inference
print("Best FADE-ConvLSTM loaded for inference")
 
mean_pm, std_pm = norm_stats['cpm25']
test_pm25       = np.load(os.path.join(TEST_IN, 'cpm25.npy')).astype(np.float32)
N               = test_pm25.shape[0]
test_pm25_norm  = (np.log1p(test_pm25 * 1e9) - mean_pm) / std_pm
del test_pm25; gc.collect()
 
other_keys_inf  = [k for k in MET_VARS + EMISSION_VARS if k in norm_stats]
available_keys  = [k for k in other_keys_inf
                   if os.path.exists(os.path.join(TEST_IN, f'{k}.npy'))]
print(f"Test keys available: {available_keys}")
 
all_preds   = []
INFER_BATCH = 4
 
with torch.no_grad():
    for i in tqdm(range(0, N, INFER_BATCH)):
        batch_mets = []
        for k in available_keys:
            arr = np.load(os.path.join(TEST_IN, f'{k}.npy'), mmap_mode='r')
            slc = np.array(arr[i:i+INFER_BATCH]).astype(np.float32)
            batch_mets.append(normalize(slc, k))
 
        met_b = torch.tensor(np.stack(batch_mets, axis=2)).to(DEVICE)
        pm_b  = torch.tensor(test_pm25_norm[i:i+INFER_BATCH]).to(DEVICE)
 
        with autocast():
            pred_norm = _model(pm_b, met_b).float().cpu().numpy()  # (B,16,H,W)
 
        pred_log = pred_norm * std_pm + mean_pm
        pred     = np.clip(np.expm1(pred_log) / 1e9, 0, None)
        all_preds.append(pred)
        del pm_b, met_b, batch_mets; torch.cuda.empty_cache()
 
preds     = np.concatenate(all_preds, axis=0)          # (218, 16, 140, 124)
preds_out = preds.transpose(0, 2, 3, 1)                # (218, 140, 124, 16)
assert preds_out.shape == (218, 140, 124, 16), f"Shape mismatch: {preds_out.shape}"
 
np.save(OUTPUT, preds_out)
print(f"Saved to {OUTPUT}")
print(f"  min={preds_out.min():.3f}, max={preds_out.max():.3f}, "
      f"mean={preds_out.mean():.3f}")

Best FADE-ConvLSTM loaded for inference
Test keys available: ['q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']


100%|██████████| 55/55 [01:55<00:00,  2.09s/it]


Saved to /kaggle/working/preds.npy
  min=0.006, max=8299.950, mean=67.178
